In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
path = kagglehub.competition_download("binaryclassificationwithabankchurndataset")
print(path)
print(os.listdir(path))


/Users/muhammadaminesaev/.cache/kagglehub/competitions/binaryclassificationwithabankchurndataset
['test.csv', 'train.csv', 'sample_submission.csv']


In [3]:
train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_sub = pd.read_csv(os.path.join(path, "sample_submission.csv"))

display(train.head())
display(test.head())



,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,0,15702656.0,Nwora,567.0,France,Male,33.0,9.0,0.0,2.0,1.0,0.0,156792.89,0.0
1,1,15647965.0,Yevdokimova,628.0,France,Female,38.0,3.0,0.0,1.0,1.0,1.0,51987.99,1.0
2,2,15798834.0,Ch'iu,635.0,France,Female,29.0,3.0,0.0,2.0,1.0,1.0,113079.19,0.0
3,3,15672056.0,Hsia,681.0,France,Male,28.0,6.0,0.0,2.0,1.0,0.0,14081.64,0.0
4,4,15759537.0,Okwudilichukwu,587.0,France,Female,27.0,5.0,0.0,2.0,1.0,0.0,158958.90,0.0


,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,15000,15732563.0,Hs?,707.0,Germany,Male,21.0,4.0,114513.66,2.0,1.0,0.0,176692.87
1,15001,15727041.0,Ma,678.0,Germany,Male,33.0,5.0,142510.50,1.0,0.0,0.0,52820.13
2,15002,15753679.0,K?,636.0,Spain,Male,45.0,9.0,0.00,2.0,1.0,1.0,125062.02
3,15003,15581554.0,H?,697.0,Spain,Male,46.0,8.0,0.00,2.0,1.0,0.0,131647.41
4,15004,15773723.0,Bellucci,553.0,Spain,Male,31.0,2.0,0.00,2.0,1.0,0.0,58814.41


In [4]:
print("train:", train.shape)
print("test:", test.shape)
print(train.isnull().sum())


train: (15000, 14)
test: (10000, 13)
id                 0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


In [5]:
print(train["Exited"].value_counts())
print(train["Exited"].value_counts(normalize=True))

sns.countplot(x="Exited", data=train)
plt.title("Stayed (0) vs Left (1)")
plt.show()


Exited
0.0    12023
1.0     2977
Name: count, dtype: int64
Exited
0.0    0.801533
1.0    0.198467
Name: proportion, dtype: float64


In [6]:
pd.DataFrame({
    "dtype": train.dtypes,
    "n_unique": train.nunique()
})


,dtype,n_unique
id,int64,15000
CustomerId,float64,6306
Surname,object,758
CreditScore,float64,378
Geography,object,3
Gender,object,2
Age,float64,55
Tenure,float64,11
Balance,float64,3328
NumOfProducts,float64,5


In [7]:
print(train["NumOfProducts"].value_counts().sort_index())
print()
print("Customers with Balance == 0:", (train["Balance"] == 0).mean())


NumOfProducts
1.0    6379
2.0    8406
3.0     194
4.0      20
5.0       1
Name: count, dtype: int64

Customers with Balance == 0: 0.6534666666666666


In [8]:
for col in ["Geography", "Gender", "NumOfProducts", "IsActiveMember", "HasCrCard"]:
    summary = train.groupby(col)["Exited"].agg(["mean", "count"])
    summary.columns = ["churn_rate", "customers"]
    print(summary.round(3))
    print()


           churn_rate  customers
Geography                       
France          0.154       9040
Germany         0.406       2680
Spain           0.152       3280

        churn_rate  customers
Gender                       
Female       0.276       6522
Male         0.139       8478

               churn_rate  customers
NumOfProducts                       
1.0                 0.380       6379
2.0                 0.042       8406
3.0                 0.954        194
4.0                 0.850         20
5.0                 1.000          1



                churn_rate  customers
IsActiveMember                       
0.0                  0.273       7560
1.0                  0.123       7440

           churn_rate  customers
HasCrCard                       
0.0             0.206       3313
1.0             0.196      11687



In [9]:
num_cols = ["Age", "CreditScore", "Balance", "EstimatedSalary"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, num_cols):
    sns.kdeplot(data=train, x=col, hue="Exited", common_norm=False, fill=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


## CustomerId and duplicates
`CustomerId` has far fewer unique values than rows. If it identified real customers, the same person could land in both
training and validation folds and inflate the CV score. Check before trusting the CV.

In [10]:
feature_cols = [c for c in test.columns if c != "id"]

print("Duplicate feature rows in train:", train.duplicated(subset=feature_cols).sum())
print("Test rows identical to a train row:", test.merge(train[feature_cols].drop_duplicates(), on=feature_cols).shape[0])
print("Test CustomerIds also in train:", round(test["CustomerId"].isin(train["CustomerId"]).mean(), 3))
print()
print("Rows per CustomerId:")
print(train["CustomerId"].value_counts().describe().round(2))

Duplicate feature rows in train: 0
Test rows identical to a train row: 0
Test CustomerIds also in train: 0.807

Rows per CustomerId:
count    6306.00
mean        2.38
std         1.93
min         1.00
25%         1.00
50%         2.00
75%         3.00
max        23.00
Name: count, dtype: float64


In [11]:
# If CustomerId were a real person, repeated rows would share Age, Surname and label.
repeated = train[train["CustomerId"].duplicated(keep=False)].groupby("CustomerId")
print("Repeated ids with different Age:    ", round((repeated["Age"].nunique() > 1).mean(), 3))
print("Repeated ids with different Surname:", round((repeated["Surname"].nunique() > 1).mean(), 3))
print("Repeated ids with different Exited: ", round((repeated["Exited"].nunique() > 1).mean(), 3))

Repeated ids with different Age:     0.982
Repeated ids with different Surname: 0.999
Repeated ids with different Exited:  0.492


**Takeaway:** no duplicate rows, and repeated `CustomerId`s mostly have different ages, surnames and labels, so the id is a
synthetic artifact, not a real person. Grouped CV (below) gives the same score as plain stratified CV, so there is no leak
and `StratifiedKFold` is fine.

## Feature engineering
Based on the EDA: churn is U-shaped in `NumOfProducts`, 65% of customers have zero balance, and churn climbs with age.

In [12]:
def add_features(df):
    df = df.copy()
    df["ZeroBalance"] = (df["Balance"] == 0).astype(int)
    df["IsSenior"] = (df["Age"] >= 60).astype(int)
    df["AgeBin"] = pd.cut(df["Age"], bins=[0, 30, 40, 50, 60, 200], labels=False)
    df["ActiveByProducts"] = df["IsActiveMember"] * df["NumOfProducts"]
    df["BalancePerProduct"] = df["Balance"] / df["NumOfProducts"]
    return df

drop_cols = ["id", "CustomerId", "Surname"]

X = add_features(train.drop(columns=drop_cols + ["Exited"]))
y = train["Exited"].astype(int)
X_test = add_features(test.drop(columns=drop_cols))

print(X.shape, X_test.shape)
X.head()

(15000, 15) (10000, 15)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,ZeroBalance,IsSenior,AgeBin,ActiveByProducts,BalancePerProduct
0,567.0,France,Male,33.0,9.0,0.0,2.0,1.0,0.0,156792.89,1,0,1,0.0,0.0
1,628.0,France,Female,38.0,3.0,0.0,1.0,1.0,1.0,51987.99,1,0,1,1.0,0.0
2,635.0,France,Female,29.0,3.0,0.0,2.0,1.0,1.0,113079.19,1,0,0,2.0,0.0
3,681.0,France,Male,28.0,6.0,0.0,2.0,1.0,0.0,14081.64,1,0,0,0.0,0.0
4,587.0,France,Female,27.0,5.0,0.0,2.0,1.0,0.0,158958.90,1,0,0,0.0,0.0


## Models
Encoding lives inside each pipeline, so it is fit on the training folds only and train/test always get the same columns
(unseen categories are ignored instead of shifting columns).

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def report(name, scores):
    print(f"{name}: AUC {scores.mean():.4f} ± {scores.std():.4f}  folds={scores.round(4)}")

In [14]:
# Logistic regression: one-hot NumOfProducts so the model can learn its U-shaped effect.
lr_cat = ["Geography", "Gender", "NumOfProducts"]
lr_num = [c for c in X.columns if c not in lr_cat]

logreg = make_pipeline(
    ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), lr_cat),
        ("num", StandardScaler(), lr_num),
    ]),
    LogisticRegression(max_iter=2000),
)
report("LogReg", cross_val_score(logreg, X, y, cv=cv, scoring="roc_auc"))

LogReg: AUC 0.9275 ± 0.0035  folds=[0.9339 0.9233 0.9267 0.9259 0.9278]


In [15]:
# HistGB handles categoricals natively; it only needs them as integer codes.
hgb_cat = ["Geography", "Gender"]

hgb = make_pipeline(
    ColumnTransformer(
        [("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), hgb_cat)],
        remainder="passthrough",
        verbose_feature_names_out=False,
    ).set_output(transform="pandas"),
    HistGradientBoostingClassifier(categorical_features=hgb_cat, random_state=42),
)
report("HistGB", cross_val_score(hgb, X, y, cv=cv, scoring="roc_auc"))

HistGB: AUC 0.9299 ± 0.0038  folds=[0.9352 0.9272 0.9293 0.9249 0.9331]


In [16]:
# Sanity check: keep all rows of a CustomerId in the same fold. Same score => no leak through repeated ids.
group_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
report("HistGB (grouped by CustomerId)",
       cross_val_score(hgb, X, y, cv=group_cv, groups=train["CustomerId"], scoring="roc_auc"))

HistGB (grouped by CustomerId): AUC 0.9310 ± 0.0048  folds=[0.9341 0.9387 0.9298 0.9261 0.9265]


## Submission

In [17]:
hgb.fit(X, y)
pred = hgb.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({"id": test["id"], "Exited": pred})
submission.to_csv("submission.csv", index=False)

print(submission.shape)
submission.head()

(10000, 2)


,id,Exited
0,15000,0.029084
1,15001,0.132665
2,15002,0.022565
3,15003,0.084499
4,15004,0.007313
